# SOXX IBKR Historical 1-min QA — v0.1.1
**Read-only source, synthetic tests only in GitHub Actions, real data in Colab.** SOXX (conId 12658194), not SOX. v0.1.1 adds persistent, verifiable checkpoint/resume, observed-only PRE/POST diagnostics and missing full trading-day audit. Start with MAX_CHUNKS=5; set 0 only after pilot. These are provisional QA reports, **not research-ready**.

Checkpoint files live in your mounted My Drive, separate from timestamped immutable result folders. Do not delete checkpoint unless you want a full re-validation. The source folder is accessed by authenticated Google Drive API, not by an assumed path in My Drive.


In [ ]:
import sys, subprocess, importlib
from pathlib import Path

REPO = Path("/content/IBKR-market-data-cleaning-pipeline")
URL = "https://github.com/hh4832/IBKR-market-data-cleaning-pipeline.git"
SRC = REPO / "src"
print("Python:", sys.version)
assert sys.version_info >= (3, 11), "Python >=3.11 required"
if (REPO / ".git").is_dir():
    origin = subprocess.check_output(
        ["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True
    ).strip()
    assert origin.rstrip("/").removesuffix(".git") == URL.removesuffix(".git"), (
        f"Unexpected repository origin: {origin}"
    )
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", "main"], check=True)
elif REPO.exists():
    raise RuntimeError(f"Existing path is not a Git repository: {REPO}")
else:
    subprocess.run(["git", "clone", "--depth", "1", URL, str(REPO)], check=True)

package_file = SRC / "ibkr_qc" / "__init__.py"
assert package_file.is_file(), f"Missing expected Python package: {package_file}"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[colab]"], check=True)
# .pth files from editable installs may not be visible until a kernel restart.
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
# A git pull changes files on disk, not objects imported in a live Colab kernel.
# Clear ONLY this project's cached modules before importing the new version.
stale_modules = [
    name for name in tuple(sys.modules)
    if name == "ibkr_qc" or name.startswith("ibkr_qc.")
]
for name in sorted(stale_modules, key=len, reverse=True):
    del sys.modules[name]
importlib.invalidate_caches()
import ibkr_qc
from ibkr_qc.drive_io import authorized_drive, list_parquet_files
from ibkr_qc.pipeline import Config, run
import inspect
required_config_args = {"audit_start_ny", "audit_end_ny"}
actual_config_args = set(inspect.signature(Config).parameters)
if not required_config_args.issubset(actual_config_args):
    raise RuntimeError(
        "STALE_IBKR_QC_CONFIG: restart the Colab runtime and run cells top to bottom. "
        f"Config loaded from {inspect.getfile(Config)}; "
        f"missing {sorted(required_config_args - actual_config_args)}"
    )
print("ibkr_qc:", ibkr_qc.__version__, ibkr_qc.__file__)
print("Config:", inspect.signature(Config))
print("HEAD:", subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())


In [ ]:
from pathlib import Path
from google.colab import drive
# Persistent My Drive directory: checkpoint survives Colab Runtime restart.
drive.mount("/content/drive")
SOURCE_FOLDER_ID = "1lIq9fWpZHy9RR2xkTIOeEAcmPniwXlNK"  # SOXX chunks, read only
OUTPUT_PARENT_ID = "1k4kZ6kVjkfn99FW7AgRC6e9MDk848oVt"  # new immutable result folders
MAX_CHUNKS = 5  # PILOT. Explicitly change to 0 for FULL after review.
ASSUME_NAIVE_UTC = False  # Do NOT guess timestamp units / timezone.
# Default coverage window is first/last observed New York date. Set these only
# when you want to audit dates outside the observed source span as well.
AUDIT_START_NY = None  # e.g. "2016-10-18" after verifying intended boundary
AUDIT_END_NY = None    # e.g. "2026-10-16" after verifying intended boundary
CACHE_DIR = Path("/content/drive/MyDrive/IBKR_SOXX_QC_PERSISTENT_CACHE_V011")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
assert str(CACHE_DIR).startswith("/content/drive/MyDrive/"), "Use persistent Drive cache for resume"
LOCAL_OUTPUT_DIR = Path("/content/soxx_qc_outputs")
print("Source:", SOURCE_FOLDER_ID, "Output:", OUTPUT_PARENT_ID)
print("Chunk scope:", MAX_CHUNKS, "Checkpoint:", CACHE_DIR)


In [ ]:
from ibkr_qc.drive_io import authorized_drive, list_parquet_files, filename_collision_audit
from ibkr_qc.pipeline import Config, run
import subprocess
service = authorized_drive()
files = list_parquet_files(service, SOURCE_FOLDER_ID)
print(f"Found {len(files)} SOXX Parquet file IDs in source folder.")
collisions = filename_collision_audit(files)
unresolved = [c for c in collisions if c["state"] != "BYTE_IDENTICAL_BY_DRIVE_MD5"]
print(f"Same-name groups: {len(collisions)}; unresolved groups: {len(unresolved)}")
for c in collisions[:10]:
    print(f"  {c['source_name']}: {c['file_count']} files, {c['state']}, IDs={c['drive_file_ids']}")
if len(collisions) > 10:
    print(f"  ... {len(collisions) - 10} additional same-name groups; see QA CSV after run")
print("First five:",[(f["name"], f["id"]) for f in files[:5]])
assert files, "No raw Parquet chunks found; check OAuth and source ID"

In [ ]:
# Refresh Config from the actual on-disk pipeline module even if the kernel
# previously imported v0.1.0. Prevents a stale dataclass constructor.
import inspect, importlib
import ibkr_qc.pipeline as _pipeline
_pipeline = importlib.reload(_pipeline)
Config, run = _pipeline.Config, _pipeline.run
if not {"audit_start_ny", "audit_end_ny"}.issubset(inspect.signature(Config).parameters):
    raise RuntimeError("STALE_IBKR_QC_CONFIG: run the first bootstrap cell, or restart Runtime and Run all")
repo = "/content/IBKR-market-data-cleaning-pipeline"
head = subprocess.check_output(["git", "-C", repo, "rev-parse", "HEAD"], text=True).strip()
print("Git SHA:", head)
config = Config(output_dir=LOCAL_OUTPUT_DIR, drive_source_id=SOURCE_FOLDER_ID,
                drive_output_id=OUTPUT_PARENT_ID, cache_dir=CACHE_DIR,
                max_chunks=MAX_CHUNKS, assume_naive_utc=ASSUME_NAIVE_UTC, git_sha=head,
                audit_start_ny=AUDIT_START_NY, audit_end_ny=AUDIT_END_NY)
out = run(config, service=service)
print("Local results:", out)


In [ ]:
import pandas as pd, json
info = json.loads((out / "run_info.json").read_text())
print(json.dumps(info, ensure_ascii=False, indent=2))
print("Checkpoint resumed:", info["checkpoint_reused_chunks"],
      "newly validated:", info["new_normalized_chunks"])
print("RTH calendar coverage:")
display(pd.read_csv(out / "daily_coverage.csv").head(30))
print("Days with no RTH bars:")
display(pd.read_csv(out / "trading_day_gaps.csv").head(30))
print("Extended-hours observed-only diagnostics (PRE/POST):")
display(pd.read_csv(out / "extended_hours_daily.csv").head(30))
print("Chunk-level QA:")
display(pd.read_csv(out / "chunk_quality.csv").tail(15))
print("Drive output:", info.get("drive_run_url", "not uploaded"))
assert info["research_ready"] is False
